- https://github.com/Shenzhi-Wang/Beyond-the-80-20-Rule-RLVR/blob/main/verl/workers/actor/dp_actor.py#L375

In [ ]:
@GPUMemoryLogger(role="dp actor", logger=logger)
    def update_policy(self, data: DataProto):
        # make sure we are in training mode
        self.actor_module.train()

        temperature = data.meta_info["temperature"]  # temperature must be in the data.meta_info to avoid silent error

        select_keys = [
            "responses",
            "response_mask",
            "input_ids",
            "attention_mask",
            "position_ids",
            "old_log_probs",
            "advantages",
        ]
        if self.config.use_kl_loss:
            select_keys.append("ref_log_prob")
        # Include pre-computed IS weights if present in batch
        # Weights are computed centrally in trainer and added to batch when algorithm.rollout_is=True
        if "rollout_is_weights" in data.batch.keys():
            select_keys.append("rollout_is_weights")

        has_multi_modal_inputs = "multi_modal_inputs" in data.non_tensor_batch.keys()
        non_tensor_select_keys = ["multi_modal_inputs"] if has_multi_modal_inputs else []

        data = data.select(batch_keys=select_keys, non_tensor_batch_keys=non_tensor_select_keys)

        # Split to make minibatch iterator for updating the actor
        # See PPO paper for details. https://arxiv.org/abs/1707.06347
        mini_batches = data.split(self.config.ppo_mini_batch_size)

        on_policy = len(mini_batches) == 1 and self.config.ppo_epochs == 1

        metrics = {}
        for _ in range(self.config.ppo_epochs):
            for batch_idx, mini_batch in enumerate(mini_batches):
                if self.config.use_dynamic_bsz:
                    max_token_len = self.config.ppo_max_token_len_per_gpu * self.ulysses_sequence_parallel_size
                    micro_batches, _ = prepare_dynamic_batch(mini_batch, max_token_len=max_token_len)
                else:
                    self.gradient_accumulation = (
                        self.config.ppo_mini_batch_size // self.config.ppo_micro_batch_size_per_gpu
                    )
                    micro_batches = mini_batch.split(self.config.ppo_micro_batch_size_per_gpu)

                self.actor_optimizer.zero_grad()

                for micro_batch in micro_batches:
                    micro_batch = micro_batch.to(get_device_id())
                    micro_batch_metrics = {}
                    model_inputs = {**micro_batch.batch, **micro_batch.non_tensor_batch}
                    response_mask = model_inputs["response_mask"]
                    old_log_prob = model_inputs["old_log_probs"]
                    advantages = model_inputs["advantages"]

                    entropy_coeff = self.config.entropy_coeff
                    loss_agg_mode = self.config.loss_agg_mode

                    if self.config.use_dynamic_bsz:
                        loss_scale_factor = response_mask.shape[0] / self.config.ppo_mini_batch_size
                    else:
                        loss_scale_factor = 1 / self.gradient_accumulation

                    # all return: (bsz, response_length)
                    calculate_entropy = False
                    entropy_top_ratio = self.config.get('entropy_top_ratio', None)
                    if entropy_top_ratio is not None and not (0 <= entropy_top_ratio <= 1):
                        raise ValueError(f"Invalid {entropy_top_ratio=}")
                    
                    if entropy_coeff != 0 or entropy_top_ratio is not None:
                        calculate_entropy = True
                    entropy, log_prob = self._forward_micro_batch(
                        model_inputs, temperature=temperature, calculate_entropy=calculate_entropy
                    )

                    # for fully_async_policy recipe
                    if hasattr(self.config, "use_rollout_log_probs") and self.config.use_rollout_log_probs:
                        old_log_prob = model_inputs["old_log_probs"]
                    else:
                        if on_policy:
                            old_log_prob = log_prob.detach()
                        else:
                            old_log_prob = model_inputs["old_log_probs"]
                    
                    entropy_top_mask = None
                    if entropy_top_ratio is not None:
                        entropy_top_mask = get_global_entropy_top_mask(entropy=entropy, response_mask=response_mask, top_ratio=entropy_top_ratio)

                    loss_mode = self.config.policy_loss.get("loss_mode", "vanilla")
                    # vanilla -> verl.trainer.ppo.core_algos.compute_policy_loss_vanilla

                    # Extract pre-computed rollout importance sampling weights if present
                    # Weights are computed centrally in trainer and added when algorithm.rollout_is=True
                    rollout_is_weights = model_inputs.get("rollout_is_weights", None)

                    # NOTE: Both mismatch diagnostic metrics (PPL, KL, etc.) and IS weight metrics
                    # are computed centrally in ray_trainer.py for consistency and efficiency.
                    # This ensures metrics are computed uniformly across all batches at the trainer level
                    # and avoids redundant computation across workers and micro-batches.

                    # gpg -> verl.trainer.ppo.core_algos.compute_policy_loss_gpg
                    # clip_cov -> verl.trainer.ppo.core_algos.compute_policy_loss_clip_cov
                    policy_loss_fn = get_policy_loss_fn(loss_mode)

                    # Compute policy loss (all functions return 4 values)
                    pg_loss, pg_clipfrac, ppo_kl, pg_clipfrac_lower = policy_loss_fn(
                        old_log_prob=old_log_prob,
                        log_prob=log_prob,
                        advantages=advantages,
                        response_mask=response_mask,
                        loss_agg_mode=loss_agg_mode,
                        config=self.config,
                        rollout_is_weights=rollout_is_weights,
                        entropy_top_mask=entropy_top_mask,
                    )

                    if entropy_coeff != 0:
                        entropy_loss = agg_loss(loss_mat=entropy, loss_mask=response_mask, loss_agg_mode=loss_agg_mode)

                        # compute policy loss
                        policy_loss = pg_loss - entropy_loss * entropy_coeff
                    else:
                        policy_loss = pg_loss

                    if self.config.use_kl_loss:
                        ref_log_prob = model_inputs["ref_log_prob"]
                        # compute kl loss
                        kld = kl_penalty(
                            logprob=log_prob, ref_logprob=ref_log_prob, kl_penalty=self.config.kl_loss_type
                        )
                        kl_loss = agg_loss(loss_mat=kld, loss_mask=response_mask, loss_agg_mode=loss_agg_mode)

                        policy_loss = policy_loss + kl_loss * self.config.kl_loss_coef
                        micro_batch_metrics["actor/kl_loss"] = kl_loss.detach().item() * loss_scale_factor
                        micro_batch_metrics["actor/kl_coef"] = self.config.kl_loss_coef

                    if self.config.use_dynamic_bsz:
                        # relative to the dynamic bsz
                        loss = policy_loss * loss_scale_factor
                    else:
                        loss = policy_loss * loss_scale_factor
                    loss.backward()

                    micro_batch_metrics.update(
                        {
                            "actor/pg_loss": pg_loss.detach().item() * loss_scale_factor,
                            "actor/pg_clipfrac": pg_clipfrac.detach().item(),
                            "actor/ppo_kl": ppo_kl.detach().item(),
                            "actor/pg_clipfrac_lower": pg_clipfrac_lower.detach().item(),
                        }
                    )
                    append_to_dict(metrics, micro_batch_metrics)

                grad_norm = self._optimizer_step()
                mini_batch_metrics = {"actor/grad_norm": grad_norm.detach().item()}
                append_to_dict(metrics, mini_batch_metrics)
        self.actor_optimizer.zero_grad()
        return metrics

<b>这份代码是 verl 框架中 Actor 模块的 update_policy 方法，用于执行 PPO 算法的策略更新。</b>

- 让我详细解析这段代码，并重点解释其中与 token 级别熵计算 和 高熵 token 筛选 相关的部分。

# 代码整体结构

In [ ]:
@GPUMemoryLogger(role="dp actor", logger=logger)
def update_policy(self, data: DataProto):

这是一个装饰器方法，用于：
- 记录 GPU 内存使用情况
- <font color='red'>在训练模式下更新 Actor（策略网络）的参数</font>
##### 输入：
  - DataProto 对象，包含训练数据（responses, masks, log_probs, advantages 等）
##### 输出：
  - <font color='red'>metrics 字典，包含各种训练指标</font>
  
# 数据准备阶段
##  选择需要的字段

In [ ]:
select_keys = [
    "responses",           # 生成的 token IDs
    "response_mask",       # 标记哪些位置是 response（而非 prompt）
    "input_ids",           # 完整输入（prompt + response）
    "attention_mask",      # 注意力掩码
    "position_ids",        # 位置编码
    "old_log_probs",       # 旧策略的 log_prob（用于 PPO ratio）
    "advantages",          # 每个 token 的 advantage
]

### 关键字段：
- old_log_probs：(bsz, response_length)，每个 response token 在旧策略下的 log 概率
- advantages：(bsz, response_length)，每个 token 的 advantage 值
- response_mask：(bsz, response_length)，标记有效 response 位置

## 可选字段

In [ ]:
if self.config.use_kl_loss:
    select_keys.append("ref_log_prob")  # 参考模型的 log_prob（用于 KL 惩罚）

if "rollout_is_weights" in data.batch.keys():
    select_keys.append("rollout_is_weights")  # 预计算的 IS 权重

# 核心：高熵 Token 筛选逻辑
- 这是论文《Beyond the 80/20 Rule》中 forking tokens 筛选的关键实现：


In [ ]:
# ========== 关键部分：决定是否计算熵 ==========
calculate_entropy = False
entropy_top_ratio = self.config.get('entropy_top_ratio', None)

# 验证参数范围
if entropy_top_ratio is not None and not (0 <= entropy_top_ratio <= 1):
    raise ValueError(f"Invalid {entropy_top_ratio=}")

# 如果设置了 entropy_coeff 或 entropy_top_ratio，就计算熵
if entropy_coeff != 0 or entropy_top_ratio is not None:
    calculate_entropy = True

# 前向传播，计算 log_prob 和 entropy
entropy, log_prob = self._forward_micro_batch(
    model_inputs, 
    temperature=temperature, 
    calculate_entropy=calculate_entropy
)

##  entropy_top_ratio 参数
这是论文中 ρ（rho） 参数的实现：
- entropy_top_ratio = 0.2 → <font color='red'>保留 top-20% 高熵 token（对应论文的 80/20 规则）</font>
- entropy_top_ratio = None → 不使用高熵筛选（vanilla PPO）
##  创建高熵 Token Mask


In [ ]:
entropy_top_mask = None
if entropy_top_ratio is not None:
    entropy_top_mask = get_global_entropy_top_mask(
        entropy=entropy,           # (bsz, response_length) 每个 token 的熵
        response_mask=response_mask,  # (bsz, response_length) 有效位置标记
        top_ratio=entropy_top_ratio   # 例如 0.2（保留 top-20%）
    )

### get_global_entropy_top_mask 函数（推测实现）：
-  <font color='red'>创建 mask，只保留 response 中 top-ratio 的高熵 token</font>

In [ ]:
def get_global_entropy_top_mask(entropy, response_mask, top_ratio=0.2):
    """
    创建 mask，只保留 response 中 top-ratio 的高熵 token
    
    Args:
        entropy: (bsz, seq_len) 每个 token 的熵值
        response_mask: (bsz, seq_len) 标记 response 位置
        top_ratio: 保留的高熵 token 比例（如 0.2 = 20%）
    
    Returns:
        mask: (bsz, seq_len) 二值 mask，高熵位置为 1
    """
    # 只考虑 response 部分的 token
    masked_entropy = entropy * response_mask  # 非 response 位置置为 0
    
    # 计算阈值：保留 top-ratio 的高熵 token
    # 方法：将所有有效位置的熵值展平，取 (1-top_ratio) 分位数作为阈值
    valid_entropy = masked_entropy[response_mask.bool()]
    
    if len(valid_entropy) > 0:
        # 计算阈值：例如 top_ratio=0.2，取 80th percentile
        # 即保留熵值大于 80% 分位数的 token
        threshold = torch.quantile(valid_entropy, 1 - top_ratio)
    else:
        threshold = 0.0
    
    # 创建 mask：熵值 >= 阈值 且 是 response token
    top_mask = (masked_entropy >= threshold) & (response_mask > 0)
    
    return top_mask.float()  # 返回 float mask，用于后续 loss 计算

# 策略损失计算
##  获取损失函数

In [ ]:
loss_mode = self.config.policy_loss.get("loss_mode", "vanilla")
policy_loss_fn = get_policy_loss_fn(loss_mode)

支持多种损失模式：
- <font color='red'>"vanilla"</font>：标准 PPO clip loss
- <font color='red'>"gpg"</font>：Generalized Policy Gradient
- <font color='red'>"clip_cov"</font>：带协方差调整的 clip


## 计算策略损失

In [ ]:
pg_loss, pg_clipfrac, ppo_kl, pg_clipfrac_lower = policy_loss_fn(
    old_log_prob=old_log_prob,
    log_prob=log_prob,
    advantages=advantages,
    response_mask=response_mask,
    loss_agg_mode=loss_agg_mode,
    config=self.config,
    rollout_is_weights=rollout_is_weights,
    entropy_top_mask=entropy_top_mask,  # 关键：传入高熵 token mask
)

### compute_policy_loss_vanilla 函数（推测实现，包含 entropy_top_mask）：

In [ ]:
def compute_policy_loss_vanilla(
    old_log_prob,
    log_prob,
    advantages,
    response_mask,
    loss_agg_mode,
    config,
    rollout_is_weights=None,
    entropy_top_mask=None,  # 新增参数
):
    # 1. 计算重要性采样比率
    ratio = torch.exp(log_prob - old_log_prob)
    
    # 2. PPO clip loss（逐 token）
    pg_losses1 = -advantages * ratio
    pg_losses2 = -advantages * torch.clamp(ratio, 1-cliprange, 1+cliprange)
    pg_losses = torch.maximum(pg_losses1, pg_losses2)
    
    # 3. 关键：应用高熵 token mask
    if entropy_top_mask is not None:
        # 只保留高熵 token 的 loss
        pg_losses = pg_losses * entropy_top_mask
    
    # 4. 聚合（只计算被 mask 的 token）
    if entropy_top_mask is not None:
        # 使用 entropy_top_mask 作为 loss 的 mask
        pg_loss = agg_loss(
            loss_mat=pg_losses, 
            loss_mask=entropy_top_mask,  # 用高熵 mask 替代 response_mask
            loss_agg_mode=loss_agg_mode
        )
    else:
        # 标准 PPO：使用 response_mask
        pg_loss = agg_loss(
            loss_mat=pg_losses,
            loss_mask=response_mask,
            loss_agg_mode=loss_agg_mode
        )
    
    return pg_loss, pg_clipfrac, ppo_kl, pg_clipfrac_lower

# 熵奖励（Entropy Bonus）

In [ ]:
if entropy_coeff != 0:
    # 计算熵损失（鼓励探索）
    entropy_loss = agg_loss(
        loss_mat=entropy, 
        loss_mask=response_mask, 
        loss_agg_mode=loss_agg_mode
    )
    
    # 策略损失 = PPO loss - 熵奖励 * 系数
    policy_loss = pg_loss - entropy_loss * entropy_coeff
else:
    policy_loss = pg_loss

## 注意：
- <font color='blue'>熵奖励（entropy_loss）通常使用 全部 response token（response_mask）</font>
- 而 PPO loss（pg_loss）可能只使用 高熵 token（entropy_top_mask）
这是论文的关键设计：<font color='red'>熵奖励鼓励全局探索，但策略梯度只更新高熵 token。</font>

# KL 惩罚（可选）

In [ ]:
if self.config.use_kl_loss:
    ref_log_prob = model_inputs["ref_log_prob"]
    
    # 计算 KL 散度
    kld = kl_penalty(
        logprob=log_prob, 
        ref_logprob=ref_log_prob, 
        kl_penalty=self.config.kl_loss_type
    )
    
    # KL 损失
    kl_loss = agg_loss(
        loss_mat=kld, 
        loss_mask=response_mask, 
        loss_agg_mode=loss_agg_mode
    )
    
    # 加入总损失
    policy_loss = policy_loss + kl_loss * self.config.kl_loss_coef

# 梯度更新

In [ ]:
# 反向传播
loss.backward()

# 优化器步骤（梯度裁剪 + 参数更新）
grad_norm = self._optimizer_step()

# 记录指标
metrics = {
    "actor/pg_loss": pg_loss.item(),
    "actor/pg_clipfrac": pg_clipfrac.item(),
    "actor/ppo_kl": ppo_kl.item(),
    "actor/grad_norm": grad_norm.item(),
}

# 完整流程图


In [ ]:
┌─────────────────────────────────────────────────────────────┐
│                    update_policy 流程                        │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 数据准备                                                 │
│     ├── 选择必要字段（responses, masks, old_log_probs,      │
│     │   advantages）                                         │
│     └── 可选字段（ref_log_prob, rollout_is_weights）         │
│                                                             │
│  2. 分批次（mini-batch → micro-batch）                      │
│     └── 支持动态 batch size                                 │
│                                                             │
│  3. 前向传播                                                 │
│     ├── 计算 log_prob（每个 token 的 log 概率）             │
│     └── 计算 entropy（每个 token 的熵）← 如果启用           │
│                                                             │
│  4. 高熵 Token 筛选（关键创新）                              │
│     ├── 检查 entropy_top_ratio 是否设置                      │
│     ├── 调用 get_global_entropy_top_mask()                  │
│     │   └── 计算阈值（如 80th percentile）                   │
│     │   └── 创建 mask：只保留 top-ratio 高熵 token           │
│     └── 传入 policy_loss_fn                                 │
│                                                             │
│  5. 策略损失计算                                             │
│     ├── 计算 PPO clip loss（逐 token）                      │
│     ├── 应用 entropy_top_mask（只保留高熵 token）           │
│     └── 聚合 loss                                           │
│                                                             │
│  6. 熵奖励（可选）                                           │
│     ├── 计算全局 entropy loss（全部 token）                  │
│     └── policy_loss = pg_loss - entropy_loss * coeff        │
│                                                             │
│  7. KL 惩罚（可选）                                          │
│     ├── 计算 KL 散度                                        │
│     └── policy_loss += kl_loss * kl_coef                    │
│                                                             │
│  8. 反向传播 + 优化器步骤                                    │
│     └── 记录 metrics                                        │
│                                                             │
└─────────────────────────────────────────────────────────────┘

# 与论文的对应关系

| 论文概念               | 代码实现                                                 |
| ------------------ | ---------------------------------------------------- |
| **Forking tokens** | <font color='red'>`entropy_top_mask` 标记的高熵 token  </font>                     |
| **ρ（rho）**         | `entropy_top_ratio` 配置参数                             |
| **阈值 τ**           | `get_global_entropy_top_mask` 中计算的 quantile          |
| **Equation (6)**   | `compute_policy_loss_vanilla` 中应用 `entropy_top_mask` |
| **DAPO 目标函数**      | `policy_loss_fn` 的选择（如 `"vanilla"` 或 `"gpg"`）        |


# 关键洞察
这段代码的核心创新是：

<font color='red'>通过 entropy_top_mask 实现"只更新高熵 token 的策略梯度"，这正是论文《Beyond the 80/20 Rule》中 forking tokens 机制的实现。</font>

## 设计要点：
- 熵计算：在前向传播时计算每个 token 的熵（calculate_entropy=True）
- <font color='blue'>阈值筛选：基于全局分布计算阈值（如 80th percentile）</font>
- Mask 应用：在 PPO loss 计算中只保留高熵 token 的梯度
- 熵奖励分离：熵奖励（鼓励探索）仍然作用于全部 token，但策略梯度只更新高熵 token

### 这种设计使得模型：
- 保留探索能力（通过全局熵奖励）
- 聚焦关键决策点（只更新高熵 token 的策略）
- 避免低熵 token 的干扰（这些 token 的梯度几乎为0，不需要更新）